![alt text](images/uspas.png)
# Fundamentals of Accelerator Physics and Technology
### (with Simulations and Measurements Lab)
# Computer Lab: Simulated beam transport with quadrupole focusing
##### Authors: K. Ruisard, N. Evans and V.S. Morozov

## We will be simulating beam transport in simple beamlines directly in Python with Xsuite. Questions to be turned in for credit are in **bold** and numbered.
### Python Notes:
- The code cells below build the lattices, compute Twiss functions, track centroid orbits, and make the plots directly in the notebook.
- Press shift+enter to execute a cell, or use the play button at the top of the window
- You can execute the whole notebook by using 'Run all cells' under the 'Run' tab. This will render all changes in Markdown (useful if you are entering answers directly into this worksheet)


</br>
Also helpful: Shift+right click brings up OS/browser right-click menu, can copy image or save.

----------


## 1. Setup

At injection (or at the start of a simulation), there is an optimal spot-size and divergence for the beam known as the “matched condition.” In a periodic focusing structure (ie, FODO line), the matched solution will be periodic as well.

We define the matched solution using the Twiss parameters $\beta_x$, $\beta_y$, $\alpha_x$, $\alpha_y$ (the spotsize and angular divergence are related to the Twiss parameters through the beam emittance). In this part of the exercise we will calculate matched beam properties for a simple FODO transport line and observe the difference in matched and unmatched transport.

| Parameter  | Value  |   
|---|---|
| Species  | Electron  |  
| Energy  | 1 GeV  |   
|  X emittance | $\epsilon_x = 6\ \mathrm{mm\,mrad}$  |  
|  Y emittance |  $\epsilon_y = 6\ \mathrm{mm\,mrad}$ |  
|  half-length of FODO cell | $L = 2.5\ \mathrm{m}$  |
|  Quadrupole geometric strength | $K_1 = 0.6\ \mathrm{m}^{-2}$    |
|  Quadrupole length | $L_{\mathrm{quad}} = 0.5\ \mathrm{m}$ |


The code below constructs a simple beamline composed of a sequence of FODO cells:
    - Focusing Quad
    - Drift
    - Defocusing Quad
    - Drift

  This simulation uses a matrix representation of all elements to propagate the Twiss parameters $\beta_x$, $\beta_y$, $\alpha_x$, $\alpha_y$, as well as propagate centroid orbits and beam-size envelopes from the Twiss parameters.


In the Xsuite version, the helper functions below replace those controls: `line.twiss()` calculates the lattice functions, the emittance conversion gives RMS beam size, and optional centroid offsets show single-particle motion.

----


### Xsuite setup cell
Run this once before the exercise cells. The lattice starts halfway through the drift between quadrupoles, which reproduces the matched values quoted later in the worksheet.


In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import xtrack as xt

pio.renderers.default = "notebook"

GEOMETRIC_EMITTANCE = 6e-6  # 6 mm mrad = 6e-6 m rad


def make_fodo_line(n_cells=1, *, k1=0.6, p0c=1e9, name_prefix=""):
    """Build n FODO cells with the same geometry as the original exercise."""
    elements = []
    names = []
    prefix = f"{name_prefix}_" if name_prefix else ""

    for i_cell in range(n_cells):
        tag = f"c{i_cell + 1}"
        elements += [
            xt.Drift(length=1.0),
            xt.Quadrupole(length=0.5, k1=k1),
            xt.Drift(length=2.0),
            xt.Quadrupole(length=0.5, k1=-k1),
            xt.Drift(length=1.0),
        ]
        names += [
            f"{prefix}d1_{tag}",
            f"{prefix}qf_{tag}",
            f"{prefix}d2_{tag}",
            f"{prefix}qd_{tag}",
            f"{prefix}d3_{tag}",
        ]

    line = xt.Line(elements=elements, element_names=names)
    line.particle_ref = xt.Particles(
        p0c=p0c,
        mass0=xt.ELECTRON_MASS_EV,
        q0=-1,
    )
    line.build_tracker()
    return line


def make_hybrid_line(first_cells=10, second_cells=10, *, k1_first=0.6, k1_second=0.5):
    elements = []
    names = []
    for source in [
        make_fodo_line(first_cells, k1=k1_first, name_prefix="strong"),
        make_fodo_line(second_cells, k1=k1_second, name_prefix="weak"),
    ]:
        for name in source.element_names:
            elements.append(source[name].copy())
            names.append(name)
    line = xt.Line(elements=elements, element_names=names)
    line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)
    line.build_tracker()
    return line


def make_match_section():
    elements = [xt.Drift(length=1.0)]
    names = ["dm0"]
    for i_quad, sign in enumerate([1, -1, 1, -1], start=1):
        elements += [xt.Quadrupole(length=0.5, k1=0), xt.Drift(length=2.0)]
        names += [f"qm{i_quad}", f"dm{i_quad}"]

    line = xt.Line(elements=elements, element_names=names)
    line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)
    for i_quad, sign in enumerate([1, -1, 1, -1], start=1):
        knob = f"kqm{i_quad}"
        line.vars[knob] = sign * 0.6
        line.element_refs[f"qm{i_quad}"].k1 = line.vars[knob]
    line.build_tracker()
    return line


def twiss_dataframe(tw):
    return tw.to_pandas()[["name", "s", "betx", "bety", "alfx", "alfy", "mux", "muy"]]


def add_sigmas(tw, emit_x=GEOMETRIC_EMITTANCE, emit_y=GEOMETRIC_EMITTANCE):
    df = tw.to_pandas()[["name", "s", "betx", "bety", "alfx", "alfy", "mux", "muy"]].copy()
    df["sigma_x_mm"] = 1e3 * np.sqrt(df["betx"] * emit_x)
    df["sigma_y_mm"] = 1e3 * np.sqrt(df["bety"] * emit_y)
    df["sigma_ratio_x_over_y"] = df["sigma_x_mm"] / df["sigma_y_mm"]
    return df


def summarize_twiss(tw, label="line"):
    sig = add_sigmas(tw)
    try:
        qx = tw.qx
        qy = tw.qy
    except AttributeError:
        qx = np.nan
        qy = np.nan
    return pd.DataFrame({
        "quantity": [
            "length [m]", "Qx", "Qy", "phase x [deg]", "phase y [deg]",
            "min betx [m]", "max betx [m]", "min bety [m]", "max bety [m]",
            "mean sigma_x [mm]", "mean sigma_y [mm]",
            "min sigma_x [mm]", "max sigma_x [mm]",
            "min sigma_y [mm]", "max sigma_y [mm]",
            "max sigma_x/sigma_y",
        ],
        label: [
            tw.s[-1], qx, qy, 360 * qx, 360 * qy,
            np.min(tw.betx), np.max(tw.betx), np.min(tw.bety), np.max(tw.bety),
            sig["sigma_x_mm"].mean(), sig["sigma_y_mm"].mean(),
            sig["sigma_x_mm"].min(), sig["sigma_x_mm"].max(),
            sig["sigma_y_mm"].min(), sig["sigma_y_mm"].max(),
            sig["sigma_ratio_x_over_y"].max(),
        ],
    })


def _hover_trace(x, y, text, name, mode="lines+markers"):
    return go.Scatter(
        x=x,
        y=y,
        mode=mode,
        name=name,
        text=text,
        hovertemplate="%{text}<br>s = %{x:.4g} m<br>value = %{y:.5g}<extra></extra>",
    )


def plot_twiss(tw, title="Twiss functions"):
    fig = go.Figure()
    names = np.asarray(tw.name, dtype=str)
    fig.add_trace(_hover_trace(tw.s, tw.betx, names, r"βx"))
    fig.add_trace(_hover_trace(tw.s, tw.bety, names, r"βy"))
    fig.update_layout(
        title=title,
        xaxis_title="s [m]",
        yaxis_title="β [m]",
        hovermode="closest",
        template="plotly_white",
        width=850,
        height=430,
    )
    fig.show()
    return fig


def plot_sigmas(tw, title="RMS beam size"):
    df = add_sigmas(tw)
    fig = go.Figure()
    fig.add_trace(_hover_trace(df["s"], df["sigma_x_mm"], df["name"], r"σx"))
    fig.add_trace(_hover_trace(df["s"], df["sigma_y_mm"], df["name"], r"σy"))
    fig.update_layout(
        title=title,
        xaxis_title="s [m]",
        yaxis_title="RMS size [mm]",
        hovermode="closest",
        template="plotly_white",
        width=850,
        height=430,
    )
    fig.show()
    return fig


def location_table(tw):
    df = add_sigmas(tw)
    idx_round = (df["sigma_x_mm"] - df["sigma_y_mm"]).abs().idxmin()
    rows = [
        ("round", idx_round),
        ("max sigma_x", df["sigma_x_mm"].idxmax()),
        ("min sigma_x", df["sigma_x_mm"].idxmin()),
        ("max sigma_y", df["sigma_y_mm"].idxmax()),
        ("min sigma_y", df["sigma_y_mm"].idxmin()),
    ]
    return pd.DataFrame({
        "condition": [row[0] for row in rows],
        "s [m]": [df.loc[row[1], "s"] for row in rows],
        "element": [df.loc[row[1], "name"] for row in rows],
    })

print(f"xtrack version: {xt.__version__}")
print(f"plotly renderer: {pio.renderers.default}")


## 2. Beamline Matching
### A) Unmatched Beam

Initially, the lattice functions are unmatched. We start with $\beta_x = \beta_y =$ 4 m, $\alpha_x = \alpha_y = 0$ mid-cell (between quadrupoles).

Run the next cell. It propagates the specified initial Twiss parameters through one cell and plots the unmatched lattice functions and RMS sizes.


In [ ]:
fodo_cell = make_fodo_line(n_cells=1, k1=0.6)

tw_unmatched_cell = fodo_cell.twiss(
    method="4d",
    betx=4.0, alfx=0.0,
    bety=4.0, alfy=0.0,
)

plot_twiss(tw_unmatched_cell, "Unmatched Twiss functions through one FODO cell");
plot_sigmas(tw_unmatched_cell, "Unmatched RMS beam size through one FODO cell");
twiss_dataframe(tw_unmatched_cell)


### B) Solving for matched solution

Run the next matched-cell cell. Xsuite finds the periodic Twiss solution for one pass through `FODOcell`.

From the matched lattice function, we can calculate phase advance:
- $\psi_x=\int \frac{ds}{\beta_x(s)}$
- Xsuite calculates this for you as `qx`, `qy`, `mux`, and `muy`.

**Q0) In the next cell, calculate the X and Y phase advances for the single FODO cell. (Search for nux and nuy under “Output Parameters” and recall $\psi=\nu*2\pi$)**

Xsuite reports `qx` and `qy` as tune-like phase advances in turns per pass. For this one-cell line, one pass is one FODO cell. Multiply by $2\pi$ only when you want radians.


In [ ]:
tw_cell = fodo_cell.twiss(method="4d")

phase_summary = pd.DataFrame({
    "quantity": ["Qx per cell", "Qy per cell", "phase x [rad]", "phase y [rad]", "phase x [deg]", "phase y [deg]"],
    "value": [tw_cell.qx, tw_cell.qy, 2 * np.pi * tw_cell.qx, 2 * np.pi * tw_cell.qy, 360 * tw_cell.qx, 360 * tw_cell.qy],
})

plot_twiss(tw_cell, "Matched Twiss functions for one FODO cell");
phase_summary


$\psi_x$ =

$\psi_y$ =


These are the “phase advance per cell,” which is an important metric for characterizing transport properties of any periodic lattice. The phase advance has to be chosen to avoid instabilities and resonant conditions.
Syphers exercise 3.12 derives these expressions for maximum and minimum betatron function for a FODO lattice in the thin-lens approximation:

$\beta_{max}=L \frac{1 + sin( \psi/2)}{sin \psi}$

$\beta_{min}=L \frac{1 - sin(\psi/2)}{sin \psi}$


Answer the following questions:


**Q1) For this cell, calculate $\beta_{min}$ and $\beta_{max}$ in two ways:**
- A) thin lens prediction
- B) using Xsuite (look at the table and plots below)

The answer to (A) and (B) should be quite close, but slightly different.

**Q2) If you increased the length of the quadrupole elements while holding both the cell length L and the phase advance $\psi$ fixed, will the difference between (A) and (B) get larger or smaller?  Explain your reasoning.**

Note: by keeping L and $\psi$ fixed, we fix the average focusing strength per unit length.



In [ ]:
psi = 2 * np.pi * tw_cell.qx
half_cell_length = 2.5
thin_lens = pd.DataFrame({
    "quantity": ["beta_min thin lens [m]", "beta_max thin lens [m]", "beta_min Xsuite [m]", "beta_max Xsuite [m]"],
    "value": [
        half_cell_length * (1 - np.sin(psi / 2)) / np.sin(psi),
        half_cell_length * (1 + np.sin(psi / 2)) / np.sin(psi),
        np.min(tw_cell.betx),
        np.max(tw_cell.betx),
    ],
})
thin_lens


**Q3) What are the average, max and min RMS beam spot sizes for a matched beam in this lattice?**
- Use the calculated Twiss parameters and recall that transverse size is $\sigma_x=\sqrt{\beta_x \epsilon_x}$
- Hint: use the tables printed by the code cells below.


In [ ]:
cell_size_summary = summarize_twiss(tw_cell, "matched FODOcell")
sigma_cell = add_sigmas(tw_cell)

plot_sigmas(tw_cell, "Matched RMS beam size in one FODO cell");
cell_size_summary




| X dimension           | Value |   Y dimension          | Value|
|-----------------------|-------|------------------------|------|
| $\langle \sigma_x \rangle_s$        | ..... |   $\langle \sigma_y \rangle_s$       | .... |
| max $\sigma_x(s)$  | ..... |  max $\sigma_y(s)$  | .... |
| min $\sigma_x(s)$ | ..... |  min $\sigma_y(s)$ | .... |
| max $\left(\sigma_x/\sigma_y\right)$ | ..... |


**Q4) Where along the lattice is the beam round? Where is it largest and smallest in the horizontal plane? In the vertical plane? Give your answer in s (meters) and in terms of elements (Focusing quad, de-focusing quad, drift)**

| condition                     | location in s (meters) |   Lattice element      |
|-----------------------        |-------                 |------------------------|
| round ($\sigma_x = \sigma_y$) | ...                    |   ...                  |
| max $\sigma_x(s)$           | ...                    |   ...                  |
| min $\sigma_x(s)$           | ...                    |   ...                  |
| max $\sigma_y(s)$           | ...                    |   ...                  |
| min $\sigma_y(s)$           | ...                    |   ...                  |


This variation of the matched envelope size along the beamline is characteristic of alternating gradient quadrupole focusing. There are other patterns used for quadrupole focusing, but the FODO arrangement is popular due to simplicity and good efficiency of focusing (small beam size per applied quadrupole field strength).




In [ ]:
location_table(tw_cell)


### C) Matched beam propagation down FODO beamline

Now extend your simulation. The lattice “FODObeamline” has 20 repetitions of the same FODO cell.

Run the long-beamline cell below. It builds 20 copies of the same FODO cell, computes the matched optics, and plots the periodic envelope over 100 m.

**Q5) Confirm that the tune of this lattice is consistent with the 1-cell solution:**    


In [ ]:
fodo_100m = make_fodo_line(n_cells=20, k1=0.6)
tw_100m = fodo_100m.twiss(method="4d")

long_line_summary = pd.DataFrame({
    "quantity": ["Qx over 100 m", "Qy over 100 m", "phase x [deg]", "phase y [deg]", "Qx / 20", "Qy / 20"],
    "value": [tw_100m.qx, tw_100m.qy, 360 * tw_100m.qx, 360 * tw_100m.qy, tw_100m.qx / 20, tw_100m.qy / 20],
})

plot_twiss(tw_100m, "Matched Twiss functions over 20 FODO cells");
plot_sigmas(tw_100m, "Matched RMS beam size over 20 FODO cells");
long_line_summary


Tune and phase advance over 100 meters:

$\nu_x= $  

$\nu_y= $

$\psi_x = $

$\psi_y = $


Tune and phase advance for 1 cell:

$\nu_x / 20 = $  

$\nu_y / 20 = $

$\psi_x / 20 =$

$\psi_y /20 = $


**Q6) How many oscillations does a single particle make in 100 m?**

You can calculate this easily, but you can also visualize single particle motion by giving the initial beam a centroid error. The centroid follows the equations of motion of a single particle.

The centroid-offset cell below launches a small horizontal offset and plots the resulting single-particle/centroid motion.


In [ ]:
centroid = fodo_100m.twiss(
    method="4d",
    x=1e-3, px=0.0,
    y=0.0, py=0.0,
    betx=tw_cell.betx[0], alfx=tw_cell.alfx[0],
    bety=tw_cell.bety[0], alfy=tw_cell.alfy[0],
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=centroid.s,
    y=1e3 * centroid.x,
    mode="lines+markers",
    name="x centroid",
    text=np.asarray(centroid.name, dtype=str),
    hovertemplate="%{text}<br>s = %{x:.4g} m<br>x = %{y:.5g} mm<extra></extra>",
))
fig.update_layout(
    title="Centroid motion from a 1 mm horizontal launch offset",
    xaxis_title="s [m]",
    yaxis_title="x [mm]",
    template="plotly_white",
    width=850,
    height=430,
)
fig.show();

pd.DataFrame({
    "quantity": ["single-particle oscillations over 100 m", "phase advance x over 100 m [deg]"],
    "value": [tw_100m.qx, 360 * tw_100m.qx],
})


### D) Propagation of mismatched beam

We will initialize our beam with a 10% mismatch and examine the effect this has on transport.  
The previous visualization gave a periodic solution with $\beta_x = \beta_y = 7.206$ meters and $\alpha_x = -\alpha_y = -1.146$.
- You can verify $\beta$ and $\alpha$ by interacting with the twiss_output plot (or downloading and viewing the data in CSV format).

The mismatched-envelope cell below starts the 100 m line with these Twiss values instead of the periodic matched values:

| Parameter  | Value  |   
|---|---|
|  Beta X  | 7.206 * 1.1 |  
|  Alpha X | -1.146 |  
|  Beta Y  | 7.206 * 1.1  |
|  Alpha Y | 1.146 |

You should observe that the envelope solution is not longer periodic with the cell length (5 m), but “beats”/ oscillates about the matched, stationary solution.


In [ ]:
tw_mismatch = fodo_100m.twiss(
    method="4d",
    betx=tw_cell.betx[0] * 1.1,
    alfx=tw_cell.alfx[0],
    bety=tw_cell.bety[0] * 1.1,
    alfy=tw_cell.alfy[0],
)

sig_mismatch = add_sigmas(tw_mismatch)
sig_matched = add_sigmas(tw_100m)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sig_matched["s"], y=sig_matched["sigma_x_mm"], mode="lines+markers",
    name="matched σx", text=sig_matched["name"],
    hovertemplate="%{text}<br>s = %{x:.4g} m<br>σx = %{y:.5g} mm<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=sig_mismatch["s"], y=sig_mismatch["sigma_x_mm"], mode="lines+markers",
    name="10% mismatched σx", text=sig_mismatch["name"],
    hovertemplate="%{text}<br>s = %{x:.4g} m<br>σx = %{y:.5g} mm<extra></extra>",
))
fig.update_layout(
    title="Envelope beating from a 10% beta mismatch",
    xaxis_title="s [m]", yaxis_title="RMS size [mm]",
    template="plotly_white", width=850, height=430,
)
fig.show();

pd.DataFrame({
    "quantity": ["Qx over 100 m", "smooth-approx envelope oscillations = 2 Qx"],
    "value": [tw_100m.qx, 2 * tw_100m.qx],
})


**Q7) Count the (approximate) number of envelope mismatch oscillations over the 100 meter beamline. Write your answer here:**

Remember, this is not the number of $\sigma_x$ extrema. We want to look at oscillation around the matched envelope solution. The matched solution has a period that matches the length of the FODO cell. The envelope mismatch oscillation has a much longer period.

Using the smooth approximation (constant focusing), we expect the frequency of the envelope oscillation to be twice that of single particle oscillation: $\nu_{envelope} = 2 \nu$. \
**The second part of (Q7): Does your answer from simulation agree with this theoretical prediction?**


### E) Matched solution with decreased quadrupole strength

Now we will look at what happens to the matched solution when we reduce the strength of quadrupole focusing. We will reduce the quadrupole strength but hold the emittance fixed. If we consider the RMS envelope equation:

$\frac{d^2 \sigma_x}{ds^2}=-K_x \sigma_x+  \frac{\epsilon_x^2}{\sigma_x^3}$

With weaker quadrupoles, the relative strength of the focusing term (proportional to K) against the defocusing emittance term (proportional to $\epsilon$) is less.

Predict what effect you expect the weaker focusing will have on the matched solution. Will the average beam size along the cell get larger or smaller? Will the aspect ratio between $\sigma_x$ and $\sigma_y$ inside the quadrupoles get larger or smaller?

1. The weak-focusing cell below constructs `FODOcell2` in Python. It is identical to `FODOcell` except that both quadrupole strengths are changed from $|k1|=0.6 m^{-2}$ to $|k1|=0.2 m^{-2}$.

2. Run the weak-focusing cell and compare its matched beam size and phase advance with the original cell.









In [ ]:
fodo_cell_weak = make_fodo_line(n_cells=1, k1=0.2)
tw_cell_weak = fodo_cell_weak.twiss(method="4d")

weak_summary = pd.concat([
    summarize_twiss(tw_cell, "k1 = 0.6"),
    summarize_twiss(tw_cell_weak, "k1 = 0.2").drop(columns=["quantity"]),
], axis=1)

plot_twiss(tw_cell_weak, "Matched Twiss functions for weak FODOcell2, k1 = 0.2");
plot_sigmas(tw_cell_weak, "Matched RMS beam size for weak FODOcell2, k1 = 0.2");
weak_summary


**Q8) Record the following parameters for the matched solution in the new FODO cell:**

| X dimension           | Value |   Y dimension          | Value|
|-----------------------|-------|------------------------|------|
| $\langle \sigma_x \rangle_s=$        | ..... |   $\langle \sigma_y \rangle_s=$       | ..... |
| max $\sigma_x(s)$  | ..... |  max $\sigma_y(s)$  | ..... |
| min $\sigma_x(s)$ | ..... |  min $\sigma_y(s)$ | ..... |
| max $\left(\sigma_x/\sigma_y\right)$ | ..... | | |
| $\psi_x=$ | ..... | $\psi_y =$ | ..... |



### F) Injection mismatch


In [ ]:
hybrid_line = make_hybrid_line(first_cells=10, second_cells=10, k1_first=0.6, k1_second=0.5)

tw_hybrid_from_strong_match = hybrid_line.twiss(
    method="4d",
    betx=tw_cell.betx[0], alfx=tw_cell.alfx[0],
    bety=tw_cell.bety[0], alfy=tw_cell.alfy[0],
)
tw_hybrid_periodic_100m = hybrid_line.twiss(method="4d")

fig = go.Figure()
names = np.asarray(tw_hybrid_from_strong_match.name, dtype=str)
fig.add_trace(_hover_trace(tw_hybrid_from_strong_match.s, tw_hybrid_from_strong_match.betx, names, "start matched to k1=0.6, βx"))
fig.add_trace(_hover_trace(tw_hybrid_from_strong_match.s, tw_hybrid_from_strong_match.bety, names, "start matched to k1=0.6, βy"))
fig.add_vline(x=50, line_dash="dash", line_color="black", annotation_text="transition")
fig.update_layout(
    title="Injection mismatch at transition to weaker cells",
    xaxis_title="s [m]", yaxis_title="β [m]",
    template="plotly_white", width=850, height=430,
)
fig.show();

plot_twiss(tw_hybrid_periodic_100m, "Periodic solution over the full 100 m hybrid beamline");

pd.concat([
    summarize_twiss(tw_hybrid_from_strong_match, "start matched to first section"),
    summarize_twiss(tw_hybrid_periodic_100m, "periodic over 100 m").drop(columns=["quantity"]),
], axis=1)


Accelerator complexes can contain many sections, where the focusing strength and quadrupole arrangement can be quite different between sections. We will consider a simplified version of this. We will first transport beam through a beamline comprised of 10 of our original "FODOcell," then transition into a second line comprised of 10 weaker focusing copies of "FODOcell2."  

The hybrid-beamline cell below builds 10 cells with $|k1|=0.6 m^{-2}$ followed by 10 cells with $|k1|=0.5 m^{-2}$, then starts from the matched Twiss parameters of the original cell: \
$\beta_x = \beta_y = 7.206$ meters and $\alpha_x = -\alpha_y = -1.146$


**Q9) Describe what happens as the beam transitions from the stronger to the weaker FODO cell. Is the beam matched in the first 10 cells? What about the last 10 cells?**


The same cell also asks Xsuite for the periodic solution over the full 100 m hybrid line.


**Q10) Is this new solution matched over the period $L=100$ m (the length of "FODObeamline")?\
Is this solution matched for the period $L=5$ m? (the length of a single FODO cell)?\
Does there exist a solution that is matched for period $L=5$ m in all 20 cells?**

Note that the matched solution is defined by the length over which a periodic solution is found. When accelerator scientists refer to the matched solution, it is usually implied that we mean for the shortest focusing period (e.g., the FODO cell). \
Also note that the maximum $\sigma_x$ for the matched solution from step 4 is larger than the maximum $\sigma_x$ for both matched FODO cells found above. A strong motivation for maintaining matched beams is to keep this maximum size as small as possible during beam transport.


### G) Matching section insert
We earlier studied beam transport in a FODO cell. However, an accelerator consists of not only FODO cells. There may be injection and extraction sections, experimental insertions, etc. For example, in a collider, the beam is usually much smaller at the interaction point than elsewhere in the accelerator.

The typical strategy to transition beam between sections of the accelerator while maintaining optimal transport is to add a matching section. This section contains quadrupoles whose strengths are optimized to transition the output bunch parameters $\alpha, \beta$ from the first section into the optimal (matched) parameters for the next section. A matching section can also be used to customize the beam parameters in a specific part of the accelerator.

In this exercise, we will be designing an injection section. The beam at injection is typically round and has independent distribtions of all phase-space coordinates. You have already seen from the FODO exercises that the beam distribution everywhere inside a FODO cell is not round and/or has non-zero correlation between the spatial and angular coordinates. We need to design a beamline section that takes the particle distribution at the end of the FODO cell and transforms it into a spatially round uncorrelated distribution.

The matching-section cells below build the four-quadrupole section directly in Xsuite, then vary the quadrupole strengths until the end of the section is round and uncorrelated: $\beta_x = \beta_y$, $\alpha_x = 0$, and $\alpha_y = 0$.


In [ ]:
match_section = make_match_section()
initial_from_fodo_end = dict(
    betx=float(tw_cell.betx[-1]), alfx=float(tw_cell.alfx[-1]),
    bety=float(tw_cell.bety[-1]), alfy=float(tw_cell.alfy[-1]),
)

tw_match_before = match_section.twiss(method="4d", **initial_from_fodo_end)

opt = match_section.match(
    method="4d",
    vary=[xt.Vary(f"kqm{i}", step=1e-3, limits=(-3, 3)) for i in range(1, 5)],
    targets=[
        xt.Target("alfx", 0.0, at="_end_point", tol=1e-8),
        xt.Target("alfy", 0.0, at="_end_point", tol=1e-8),
        xt.Target(lambda tw: tw["betx", "_end_point"] - tw["bety", "_end_point"], 0.0, tol=1e-8),
    ],
    **initial_from_fodo_end,
)

tw_match_after = match_section.twiss(method="4d", **initial_from_fodo_end)

fig = go.Figure()
fig.add_trace(_hover_trace(tw_match_before.s, tw_match_before.betx, np.asarray(tw_match_before.name, dtype=str), "before βx"))
fig.add_trace(_hover_trace(tw_match_before.s, tw_match_before.bety, np.asarray(tw_match_before.name, dtype=str), "before βy"))
fig.add_trace(_hover_trace(tw_match_after.s, tw_match_after.betx, np.asarray(tw_match_after.name, dtype=str), "after βx"))
fig.add_trace(_hover_trace(tw_match_after.s, tw_match_after.bety, np.asarray(tw_match_after.name, dtype=str), "after βy"))
fig.update_layout(
    title="Matching section before and after optimization",
    xaxis_title="s [m]", yaxis_title="β [m]",
    template="plotly_white", width=850, height=430,
)
fig.show();

pd.DataFrame({
    "quantity": ["final betx [m]", "final bety [m]", "final alfx", "final alfy", "QM1 k1", "QM2 k1", "QM3 k1", "QM4 k1"],
    "value": [
        tw_match_after.betx[-1], tw_match_after.bety[-1], tw_match_after.alfx[-1], tw_match_after.alfy[-1],
        match_section.vars["kqm1"]._value, match_section.vars["kqm2"]._value,
        match_section.vars["kqm3"]._value, match_section.vars["kqm4"]._value,
    ],
})


In [ ]:
# Phase-space ellipses before and after the matching section.
def ellipse_points(beta, alpha, emit=GEOMETRIC_EMITTANCE, n=300):
    phase = np.linspace(0, 2 * np.pi, n)
    x = np.sqrt(emit * beta) * np.cos(phase)
    xp = -np.sqrt(emit / beta) * (alpha * np.cos(phase) + np.sin(phase))
    return 1e3 * x, 1e3 * xp

x_in, xp_in = ellipse_points(tw_match_after.betx[0], tw_match_after.alfx[0])
x_out, xp_out = ellipse_points(tw_match_after.betx[-1], tw_match_after.alfx[-1])

y_in, yp_in = ellipse_points(tw_match_after.bety[0], tw_match_after.alfy[0])
y_out, yp_out = ellipse_points(tw_match_after.bety[-1], tw_match_after.alfy[-1])

fig = make_subplots(rows=1, cols=2, subplot_titles=("Horizontal phase-space ellipse", "Vertical phase-space ellipse"))
fig.add_trace(go.Scatter(x=x_in, y=xp_in, mode="lines", name="x input", hovertemplate="x = %{x:.5g} mm<br>x' = %{y:.5g} mrad<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=x_out, y=xp_out, mode="lines", name="x output", hovertemplate="x = %{x:.5g} mm<br>x' = %{y:.5g} mrad<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=y_in, y=yp_in, mode="lines", name="y input", hovertemplate="y = %{x:.5g} mm<br>y' = %{y:.5g} mrad<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=y_out, y=yp_out, mode="lines", name="y output", hovertemplate="y = %{x:.5g} mm<br>y' = %{y:.5g} mrad<extra></extra>"), row=1, col=2)
fig.update_xaxes(title_text="x [mm]", row=1, col=1)
fig.update_yaxes(title_text="x' [mrad]", row=1, col=1)
fig.update_xaxes(title_text="y [mm]", row=1, col=2)
fig.update_yaxes(title_text="y' [mrad]", row=1, col=2)
fig.update_layout(template="plotly_white", width=950, height=430)
fig.show();


In [ ]:
# Extra-credit optics sketch: FODO cell, optimized matching section, then the matching section in reverse order.
reverse_elements = []
reverse_names = []
for name in reversed(match_section.element_names):
    if name == "_end_point":
        continue
    reverse_elements.append(match_section[name].copy())
    reverse_names.append(f"rev_{name}")

insertion_elements = [fodo_cell[name].copy() for name in fodo_cell.element_names]
insertion_names = [f"fodo_{name}" for name in fodo_cell.element_names]
for name in match_section.element_names:
    if name == "_end_point":
        continue
    insertion_elements.append(match_section[name].copy())
    insertion_names.append(f"match_{name}")
insertion_elements += reverse_elements
insertion_names += reverse_names

insertion = xt.Line(elements=insertion_elements, element_names=insertion_names)
insertion.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)
insertion.build_tracker()

tw_insertion = insertion.twiss(method="4d", **initial_from_fodo_end)
plot_twiss(tw_insertion, "Example injection insertion optics");


**Q11) Attach graphs of the input and output xx' distributions.**
- The matching-section code cells below generate these graphs directly.

**Q12) Attach graphs of $\beta_{x,y}$ before and after optimization.**
- The matching-section code cells below generate these graphs directly.

**Q13) What are the final values of $\beta_{x,y}$ at the end of "MATCHsection"?**
- Hint: examine the table printed by the matching-section code cell.

**Q14) What are the optimized strengths of the quads in "MATCHsection"?**
- Hint: examine the table printed by the matching-section code cell.

**Q15) (Extra credit) Design an entire injection insertion and show the resulting optics.**
- The optional final cell builds an insertion from a FODO cell, the optimized `MATCHsection`, and a reversed copy of that section, then plots the resulting optics.
